# 03 - Building a Real Agent with LangGraph

Now the two notebooks come together. You will build the same tool-using agent **twice**:

1. With `create_agent` — four lines.
2. By hand as a `StateGraph` — about thirty lines.

The second version is the point. `create_agent` is a convenience wrapper that builds
exactly the graph you are about to draw yourself, and knowing that is the difference
between using an agent and being able to debug one.

> **No API key needed.** A scripted model drives the loop so you can watch every step.
> Everything works identically with a real model — swap in `real_model()`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("."))

from typing import Annotated, Literal
from typing_extensions import TypedDict

from langchain.tools import tool
from langchain.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

from offline_llm import ScriptedChatModel, ai, tool_call
print("ready")

## 1. The tools

Three tools a customer-support agent would plausibly need.

In [ ]:
@tool
def search_docs(query: str) -> str:
    """Search the internal knowledge base for a topic. Use for company policy questions."""
    kb = {
        "refund": "Refunds are issued within 14 days of purchase, no questions asked.",
        "shipping": "Standard shipping takes 3-5 business days. Express is next-day.",
        "warranty": "All hardware carries a 2-year warranty from the delivery date.",
    }
    for key, value in kb.items():
        if key in query.lower():
            return value
    return "No matching document found."

@tool
def get_order_status(order_id: str) -> str:
    """Look up the delivery status of an order by its ID."""
    orders = {"A-1001": "shipped, arriving Tuesday",
              "A-1002": "processing",
              "A-1003": "delivered on 12 March"}
    return orders.get(order_id.upper(), f"No order found with ID {order_id}")

@tool
def calculate(expression: str) -> str:
    """Evaluate a simple arithmetic expression, e.g. '119.98 * 0.15'."""
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:          # never exec() raw model output
        return "Refused: expression contains disallowed characters."
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as exc:
        return f"Error: {exc}"

TOOLS = [search_docs, get_order_status, calculate]
print("tools:", [t.name for t in TOOLS])

> **Note the guard in `calculate`.** The model chooses the string that gets evaluated.
> Never pass model output to `eval`, `exec`, `os.system` or a raw SQL string without
> validating it. Treat every tool argument as untrusted user input — because it is.

## 2. Version one — `create_agent`

In [ ]:
from langchain.agents import create_agent

# The script the stand-in model will follow: look up the order, then answer.
llm = ScriptedChatModel(responses=[
    ai("", [tool_call("get_order_status", order_id="A-1001")]),
    ai("Your order A-1001 has shipped and should arrive on Tuesday."),
])

agent = create_agent(
    model=llm,
    tools=TOOLS,
    system_prompt="You are a support agent. Use tools to look up facts. Never guess.",
)

result = agent.invoke({"messages": [HumanMessage("Where is my order A-1001?")]})

for m in result["messages"]:
    kind = type(m).__name__
    if kind == "AIMessage" and m.tool_calls:
        print(f"{kind:13} -> calls {m.tool_calls[0]['name']}({m.tool_calls[0]['args']})")
    else:
        print(f"{kind:13} {m.content}")

Four lines of setup, and the full loop ran: the model asked for a tool, LangGraph executed
it, fed the `ToolMessage` back, and the model wrote the final answer.

### What `create_agent` actually returns

A compiled LangGraph graph. Look at it.

In [ ]:
print(agent.get_graph().draw_mermaid())

Read that diagram against notebook 02: a **model** node, a **tools** node, a **conditional
edge** from the model that either goes to tools or to `END`, and an edge from tools **back**
to the model. That backward edge is the loop.

## 3. Version two — the same agent, by hand

Now build it yourself. This is the whole of "what an agent is".

In [ ]:
from langgraph.prebuilt import ToolNode

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

tools_by_name = {t.name: t for t in TOOLS}

def call_model(state: AgentState) -> dict:
    """Node 1: ask the LLM what to do next."""
    system = SystemMessage("You are a support agent. Use tools to look up facts. Never guess.")
    response = model_with_tools.invoke([system] + state["messages"])
    return {"messages": [response]}

def call_tools(state: AgentState) -> dict:
    """Node 2: run whatever the LLM asked for, return one ToolMessage per call."""
    last = state["messages"][-1]
    results = []
    for call in last.tool_calls:
        output = tools_by_name[call["name"]].invoke(call["args"])
        results.append(ToolMessage(content=str(output), tool_call_id=call["id"]))
    return {"messages": results}

def should_continue(state: AgentState) -> Literal["tools", "__end__"]:
    """The conditional edge: did the model ask for a tool, or is it done?"""
    last = state["messages"][-1]
    return "tools" if getattr(last, "tool_calls", None) else END

builder = StateGraph(AgentState)
builder.add_node("model", call_model)
builder.add_node("tools", call_tools)

builder.add_edge(START, "model")
builder.add_conditional_edges("model", should_continue)
builder.add_edge("tools", "model")               # <- the loop back

hand_built = builder.compile()
print(hand_built.get_graph().draw_mermaid())

In [ ]:
llm = ScriptedChatModel(responses=[
    ai("", [tool_call("search_docs", query="refund policy")]),
    ai("You can return it for a full refund within 14 days of purchase."),
])
model_with_tools = llm.bind_tools(TOOLS)

out = hand_built.invoke({"messages": [HumanMessage("Can I get a refund?")]})

for m in out["messages"]:
    kind = type(m).__name__
    if kind == "AIMessage" and m.tool_calls:
        print(f"{kind:13} -> {m.tool_calls[0]['name']}({m.tool_calls[0]['args']})")
    else:
        print(f"{kind:13} {m.content}")

**That is an agent.** Two nodes and a conditional edge that loops. Everything else — memory,
streaming, human approval, multi-agent — is elaboration on this skeleton.

`ToolNode` from `langgraph.prebuilt` replaces `call_tools` with a tested implementation
that also handles errors and parallel calls:

```python
builder.add_node("tools", ToolNode(TOOLS))
```

## 4. Multi-step reasoning

The loop runs as many times as the model needs. Here it uses two tools in sequence.

In [ ]:
llm = ScriptedChatModel(responses=[
    ai("", [tool_call("get_order_status", "c1", order_id="A-1003")]),
    ai("", [tool_call("calculate", "c2", expression="119.98 * 0.15")]),
    ai("Order A-1003 was delivered on 12 March. A 15% restocking fee on $119.98 "
       "is $18.00, so you would be refunded $101.98."),
])
model_with_tools = llm.bind_tools(TOOLS)

out = hand_built.invoke({"messages": [
    HumanMessage("Order A-1003 - when did it arrive, and what is 15% of $119.98?")
]})

print(f"{len(out['messages'])} messages, {sum(1 for m in out['messages'] if type(m).__name__ == 'ToolMessage')} tool calls\n")
for m in out["messages"]:
    kind = type(m).__name__
    if kind == "AIMessage" and m.tool_calls:
        print(f"  {kind:13} -> {m.tool_calls[0]['name']}({m.tool_calls[0]['args']})")
    else:
        print(f"  {kind:13} {str(m.content)[:90]}")

The graph went model → tools → model → tools → model, deciding at each pass. **Nobody wrote
that sequence.** That is what distinguishes an agent from a chain, and it is also why
agents are harder to test — the path is not fixed.

## 5. Memory

Same mechanism as notebook 02: a checkpointer plus a `thread_id`.

In [ ]:
llm = ScriptedChatModel(responses=[
    ai("", [tool_call("get_order_status", order_id="A-1002")]),
    ai("Order A-1002 is still processing."),
    ai("You asked about order A-1002 a moment ago."),
])
model_with_tools = llm.bind_tools(TOOLS)

memory_agent = builder.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "customer-42"}}

r1 = memory_agent.invoke({"messages": [HumanMessage("Status of A-1002?")]}, config=config)
print("turn 1:", r1["messages"][-1].content)

r2 = memory_agent.invoke({"messages": [HumanMessage("What did I just ask about?")]}, config=config)
print("turn 2:", r2["messages"][-1].content)

print(f"\nthread now holds {len(r2['messages'])} messages")

## 6. Streaming the agent's work

Users will not wait in silence while an agent runs three tools. Stream the steps.

In [ ]:
llm = ScriptedChatModel(responses=[
    ai("", [tool_call("search_docs", query="warranty")]),
    ai("Hardware is covered by a 2-year warranty from delivery."),
])
model_with_tools = llm.bind_tools(TOOLS)

for chunk in hand_built.stream(
        {"messages": [HumanMessage("How long is the warranty?")]},
        stream_mode="updates"):
    for node, update in chunk.items():
        msg = update["messages"][-1]
        if getattr(msg, "tool_calls", None):
            print(f"[{node}] calling {msg.tool_calls[0]['name']}...")
        else:
            print(f"[{node}] {str(msg.content)[:80]}")

## 7. Structured output from an agent

An agent that returns free text is hard to plug into other systems. `response_format`
makes the final answer a validated object.

In [ ]:
from pydantic import BaseModel, Field

class SupportAnswer(BaseModel):
    """The final structured answer from the support agent."""
    answer: str = Field(description="The reply to show the customer")
    used_tools: list[str] = Field(description="Names of tools consulted")
    needs_human: bool = Field(description="True if a human agent must follow up")

print("""With a real model:

    agent = create_agent(
        model=real_model("openai:gpt-4.1-mini"),
        tools=TOOLS,
        system_prompt="You are a support agent.",
        response_format=SupportAnswer,
    )
    result = agent.invoke({"messages": [HumanMessage("Where is order A-1001?")]})
    result["structured_response"].needs_human      # -> False

The agent runs its tool loop as usual, then fills the schema at the end.""")

## 8. `create_agent` or hand-built?

| Use `create_agent` | Build a `StateGraph` |
|---|---|
| one agent, one tool loop | several agents handing off |
| the standard ReAct pattern | custom routing before or after the model |
| you want it working today | a human approval step mid-run |
| — | non-LLM nodes (database, validation, retries) |

**Start with `create_agent`.** Drop to a `StateGraph` when you hit something it will not
express — and now you know that dropping down is a small step, not a rewrite.

---
## Exercises

1. Add a `create_ticket(summary: str, urgency: str)` tool that appends to a Python list.
   Script the model to call it, and confirm the list changed.
2. Replace the hand-written `call_tools` with `ToolNode(TOOLS)`. Confirm the output is
   identical.
3. Make one tool raise an exception. What happens with your `call_tools`? What happens with
   `ToolNode`? Which behaviour do you want in production?
4. Add a `max_steps` counter to `AgentState` and make `should_continue` end the run after
   4 tool calls. Why does every production agent need this?
5. Script the model to call **two** tools in a single `AIMessage` (pass two entries to
   `tool_calls`). Does your `call_tools` handle it? Does `ToolNode`?
6. **Stretch:** add a `validate` node between `tools` and `model` that rejects a
   `ToolMessage` longer than 500 characters and replaces it with a truncation notice.
   This is real context management — see module 07.

## Next

Notebook 04 covers the patterns that turn this into something you would actually deploy:
human approval, multi-agent handoff, error handling and observability.